In [61]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import mlflow

In [62]:
mlflow.set_experiment("Fraud_Detection_Hybrid_Model")

<Experiment: artifact_location='/home/workstation-p/Downloads/Projects/Hybrid-Approach-for-Digital-Fraud-Detection-in-Online-Financial-Transactions/mlruns/1', creation_time=1776631581517, experiment_id='1', last_update_time=1776631581517, lifecycle_stage='active', name='Fraud_Detection_Hybrid_Model', tags={}, workspace='default'>

In [20]:
def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else: df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

In [21]:
df = pd.read_csv('https://media.githubusercontent.com/media/Arannamoy/datasets/refs/heads/main/CCFD/creditcard.csv')
df = reduce_mem_usage(df)

In [22]:
df.head(5).T

,0,1,2,3,4
Time,0.000000,0.000000,1.000000,1.000000,2.000000
V1,-1.359807,1.191857,-1.358354,-0.966272,-1.158233
V2,-0.072781,0.266151,-1.340163,-0.185226,0.877737
V3,2.536347,0.166480,1.773209,1.792993,1.548718
V4,1.378155,0.448154,0.379780,-0.863291,0.403034
V5,-0.338321,0.060018,-0.503198,-0.010309,-0.407193
V6,0.462388,-0.082361,1.800499,1.247203,0.095921
V7,0.239599,-0.078803,0.791461,0.237609,0.592941
V8,0.098698,0.085102,0.247676,0.377436,-0.270533
V9,0.363787,-0.255425,-1.514654,-1.387024,0.817739


In [35]:
def train_lightgbm(X_train, y_train):
    dtrain = lgb.Dataset(X_train, label=y_train)
    params = {'objective': 'binary', 'metric': 'auc', 'device': 'gpu'} # আপনার GPU থাকলে
    model = lgb.train(params, dtrain, num_boost_round=100)
    return model

In [36]:
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = self.fc(h_n[-1])
        return self.sigmoid(out)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("LightGBM")
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'learning_rate': 0.005,
    'device': 'gpu' 
}
lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=100)
lgb_preds = lgb_model.predict(X_test)

LightGBM


In [39]:
class FraudDataset(Dataset):
    def __init__(self, X, y, window_size=10):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size + 1

    def __getitem__(self, idx):
        return self.X[idx:idx+self.window_size], self.y[idx+self.window_size-1]

In [40]:
class FraudLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(FraudLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.sigmoid(self.fc(h_n[-1]))

In [41]:
window_size = 10
input_dim = X_train.shape[1]
model_lstm = FraudLSTM(input_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_lstm.to(device)

FraudLSTM(
  (lstm): LSTM(30, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [46]:
print(f"Training LSTM on {device}...")
train_loader = DataLoader(FraudDataset(X_train, y_train, window_size), batch_size=1024, shuffle=True)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

Training LSTM on cuda...


In [52]:
model_lstm.train()
for epoch in range(50): 
    epoch_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(inputs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

Epoch 1 | Loss: 0.0031
Epoch 2 | Loss: 0.0030
Epoch 3 | Loss: 0.0029
Epoch 4 | Loss: 0.0028
Epoch 5 | Loss: 0.0028
Epoch 6 | Loss: 0.0027
Epoch 7 | Loss: 0.0026
Epoch 8 | Loss: 0.0026
Epoch 9 | Loss: 0.0025
Epoch 10 | Loss: 0.0024
Epoch 11 | Loss: 0.0024
Epoch 12 | Loss: 0.0023
Epoch 13 | Loss: 0.0022
Epoch 14 | Loss: 0.0021
Epoch 15 | Loss: 0.0020
Epoch 16 | Loss: 0.0019
Epoch 17 | Loss: 0.0018
Epoch 18 | Loss: 0.0018
Epoch 19 | Loss: 0.0016
Epoch 20 | Loss: 0.0015
Epoch 21 | Loss: 0.0013
Epoch 22 | Loss: 0.0012
Epoch 23 | Loss: 0.0011
Epoch 24 | Loss: 0.0009
Epoch 25 | Loss: 0.0009
Epoch 26 | Loss: 0.0007
Epoch 27 | Loss: 0.0006
Epoch 28 | Loss: 0.0008
Epoch 29 | Loss: 0.0006
Epoch 30 | Loss: 0.0005
Epoch 31 | Loss: 0.0004
Epoch 32 | Loss: 0.0004
Epoch 33 | Loss: 0.0003
Epoch 34 | Loss: 0.0005
Epoch 35 | Loss: 0.0003
Epoch 36 | Loss: 0.0002
Epoch 37 | Loss: 0.0002
Epoch 38 | Loss: 0.0002
Epoch 39 | Loss: 0.0001
Epoch 40 | Loss: 0.0001
Epoch 41 | Loss: 0.0001
Epoch 42 | Loss: 0.0001
E

In [53]:
test_loader = DataLoader(FraudDataset(X_test, y_test, window_size), batch_size=2048)

print("Predicting with LSTM...")
model_lstm.eval()
lstm_preds = []
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        outputs = model_lstm(inputs)
        lstm_preds.extend(outputs.cpu().squeeze().tolist())

Predicting with LSTM...


In [54]:
padding = [0] * (len(y_test) - len(lstm_preds))
lstm_preds_converted = np.array(padding + lstm_preds)

In [55]:
final_preds = (0.4 * lgb_preds) + (0.6 * lstm_preds_converted)
final_labels = (final_preds > 0.5).astype(int)

print("\n--- Final Hybrid Model Report ---")
print(classification_report(y_test, final_labels))
print(f"Hybrid AUC Score: {roc_auc_score(y_test, final_preds):.4f}")


--- Final Hybrid Model Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.91      0.73      0.81        98

    accuracy                           1.00     56962
   macro avg       0.96      0.87      0.91     56962
weighted avg       1.00      1.00      1.00     56962

Hybrid AUC Score: 0.9473


In [56]:
torch.save(model_lstm.state_dict(), 'fraud_lstm_model.pth')
lgb_model.save_model('lgb_fraud_model.txt')
print("Models saved successfully!")

Models saved successfully!
